---
format: 
    html:
      toc: true
title: "Raw data alignment"
author: "Stig U Andersen, Mikkel H Schierup, Samuele Soraggi"
date-modified: last-modified
title-block-banner: true
---

:::{.callout-note title="Tutorial description"}

This tutorial will cover the steps for performing the alignment of raw RNA- and HiFi-sequencing data. You will need to use [the software IGV](https://software.broadinstitute.org/software/igv/download) on your computer to visualize some of the output files, which can be easily downloaded once they are produced. At the end of this tutorial you will be able to:

- perform and discuss quality control on raw data in `fastq` format using `FastQC` and `MultiQC`
- align HiFi and RNA sequencing data with dedicated tools such as `MiniMap2` and `STAR`
- analyze the quality the alignment with `qualimap`

The output of this notebook will be used for the Variant calling analysis and the bulk RNA-sequencing analysis.
:::



# Biological background

White clover (*Trifolium repens*) is a common plant found in fields across the world. It has an unusual origin story: it's the result of two different plant species "merging" their genetic material. At some point during evolution (during the last ice age), these two diploid (2n) parent species—*T. occidentale* and *T. pallescens*—naturally hybridized and created white clover, which is in result allotetraploid (4n) (see figure below). 

<figure>
<img src="images/white_clover.png" width="700" alt="Kernel Choice" class="center">
</figure>

- Normal plants have two copies of each gene: one from maternal origin, one from paternal origin (these plants are *diploid*, 2n)
- White clover, however has **four copies** of each gene. That is, two copies inherited from *T. occidentale* and two from *T. pallescens* (called an *allotetraploid*, 4n)
- These four copies are in the same nucleus, you can think of it as two complete genomes side-by-side.

Normally, white clover  is an obligate outcrosser species—that means that plants swap pollen with other plants instead of fertilizing themselves. However, a special self-compatible (it *can* self-fertilize) line was used for sequencing its genome [(Griffiths et al, 2019)](https://academic.oup.com/plcell/article/31/7/1466/5985684). This line is designated as `S10` in our data (this is the 10th self-fertilized generation). In addition, we also have data from a wild clover variety (ecotype) called Tienshan (`Ti`), from mountains in China. This variety is adapted to alpine conditions, making it genetically different from the S10 line.

We have sequences (DNA "reads") from both clover varieties, and we need to align them to the white clover reference genome. However, our reference genome is tricky: it contains sequences from **both parent species** (we call them `contig 1` and `contig 2`). Therefore, when we align short DNA reads to the reference, some reads might match equally well to both `contig 1` and `contig 2` (they're similar but different species). 

We'll use quality control tools to: (1) Align reads to the **complete reference** (both contigs together); (2) Align reads to **each subgenome separately** (`contig 1` alone, `contig 2` alone); and (3) Compare the results to see how this affects our data quality and interpretation

# Quality control and mapping

## Quality Control

We run `FastQC` on the PacBio Hifi reads and on two of the Illumina RNA-seq libraries. `FastQC` does quality control of the raw sequence data, providing an overview of the data which can help identify if there are any problems that should be addressed before further analysis. You can find the report for each file into the folder `results/fastqc_output/`. The output is in HTML format and can be opened in any browser or in `jupyterlab`. It is however not easy to compare the various libraries by opening separate reports. To aggregate all the results, we apply the `MultiQC` software to the reports' folder. The output of MultiQC is in the directory `results/multiqc_output/fastqc_data`.

In [10]:
%%bash
#run fastqc
mkdir -p results/fastqc_output
fastqc -q -o results/fastqc_output ../Data/Clover_Data/*.fastq  > /dev/null 2>&1

**Note:** `fastqc` prints a lot of output conisting of a simple confirmation of execution without error, even when using the option `-q`, which means `quiet`. Therefore we added `> /dev/null 2>&1` to the command to mute the printing of that output.

In [13]:
%%bash
#run multiqc
mkdir -p results/multiqc_output/fastqc_data
multiqc --outdir results/multiqc_output/fastqc_data results/fastqc_output


/// ]8;id=12707457;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /faststorage/project/ngssummer2024/2026/manuel/Intro-NGS-AU_course/Notebooks/results/fastqc_output
         searching | ━━━━━━━━━━━━━━━━━━━━╸  98% 49/50   0% 0/50  0m results/fastqc_output/S10_1_2.R1_fastqc.html33/50 results/fastqc_output/TI_2_1.R1_fastqc.html0m results/fastqc_output/S10_2_2.R2_fastqc.html━━━━━━━━━━━━━━━━━━━━ 100% 50/50  
            fastqc | Found 25 reports
     write_results | Data        : results/multiqc_output/fastqc_data/multiqc_data
     write_results | Report      : results/multiqc_output/fastqc_data/multiqc_report.html
           multiqc | MultiQC complete


<div class="alert-success"> <font size="+2"> <b> Questions </b> </font> </div>

Visualize the report generated by MultiQC.

The report should be in `Notebooks/results/multiqc_output/fastqc_data/multiqc_report.html`

Hint: You can find a `Help` button that offers additional information about the plots for each panel. Focus on the following panels: “Per base sequence quality”, “Per sequence quality scores”.... (“Per base sequence content” always gives a `FAIL` for RNA-seq data).

* Look at the sequence quality scores: is there a marked difference between HiFi and illumina data?
* Is there anything off in the GC content? Why?

## Hifi data mapping — long DNA reads

We map the PacBio Hifi reads (`Hifi_reads_white_clover.fastq`) to the white clover reference sequence (Contig1&2) using `minimap2`.

To demonstrate how mapping algorithm choice can heavily affect results, we deliberately run the alignment twice with two different preset options (`-x` flag):

* `map-hifi`: optimized for long reads (PacBio HiFi data)
* `sr`: optimized for short reads (Illumina data) 


The `map-hifi` setting expects long reads and employ algorithms that **handle large insertions, deletions, and other structural variations more effectively**. The scoring and alignment thresholds are adjusted to account for the longer sequence context.
The `sr` setting xpect shorter reads and optimize for quick, efficient alignment of these short sequences. The **focus is on minimizing mismatches and handling the dense packing of short reads**.

The `map-hifi` setting is designed for long reads and is more lenient with gaps and mismatches typical of long-read sequencing. The `sr` setting assumes short reads and is stricter about matches. By comparing both, you can see how choosing the wrong algorithm/option can affect your results—an important lesson in bioinformatics. **know your options**.

Next, we create reports of the mapping results by running `QualiMap` on the two obtained SAM files.

We first need to index the reference fasta files using `samtools faidx`. This produces files in `.fai` format containing informations about length of the reference sequence, offset for the quality scores, name of the reference sequence. [Click here](http://www.htslib.org/doc/faidx.html) for a detailed overview. 

In [15]:
%%bash
#copy the reference data in the folder reference_data, so that you can write the indexing files
mkdir -p reference_data
cp ../Data/Clover_Data/DNA_Contig1_2.fasta ../Data/Clover_Data/DNA_Contig1.fasta ../Data/Clover_Data/DNA_Contig2.fasta reference_data

In [16]:
%%bash
samtools faidx reference_data/DNA_Contig1_2.fasta
samtools faidx reference_data/DNA_Contig1.fasta
samtools faidx reference_data/DNA_Contig2.fasta

we create an output folder for the HIFI alignment, and run `minimap2` with the settings explained before.

In [17]:
%%bash 
mkdir -p results/HIFI_alignment/
minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sam \
                            reference_data/DNA_Contig1_2.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq 

minimap2 -a -x sr -o results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sam \
                            reference_data/DNA_Contig1_2.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq

[M::mm_idx_gen::0.035*0.97] collected minimizers
[M::mm_idx_gen::0.085*1.74] sorted minimizers
[M::main::0.085*1.73] loaded/built the index for 2 target sequence(s)
[M::mm_mapopt_update::0.093*1.67] mid_occ = 50
[M::mm_idx_stat] kmer size: 19; skip: 19; is_hpc: 0; #seq: 2
[M::mm_idx_stat::0.098*1.63] distinct minimizers: 165056 (78.87% are singletons); average occurrences: 1.271; average spacing: 9.959; total length: 2089554
[M::worker_pipeline::7.788*2.74] mapped 4395 sequences
[M::main] Version: 2.31-r1302
[M::main] CMD: minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sam reference_data/DNA_Contig1_2.fasta ../Data/Clover_Data/Hifi_reads_white_clover.fastq
[M::main] Real time: 7.801 sec; CPU: 21.383 sec; Peak RSS: 1.348 GB
[M::mm_idx_gen::0.044*0.99] collected minimizers
[M::mm_idx_gen::0.067*1.49] sorted minimizers
[M::main::0.067*1.49] loaded/built the index for 2 target sequence(s)
[M::mm_mapopt_update::0.067*1.49] mid_occ = 1000
[M::mm_idx_sta

`samtools sort` is used to sort the alignment with left-to-right coordinates. The output is in `.bam` format, with `.sam` files in input (Note that you could have gotten `.bam` files from `minimap2` with a specific option).

In [18]:
%%bash
samtools sort results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sam \
                -o results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam

samtools sort results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sam \
                -o results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam

`samtools index` creates the index for the `bam` file, stored in `.bai` format. The index file lets programs access any position into the aligned data without reading the whole file, which would take too much time.

In [19]:
%%bash
samtools index results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam
samtools index results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam

Run quality control on both files

In [20]:
%%bash
qualimap bamqc -bam results/HIFI_alignment/PacBio_clover_alignment_1_2_maphifi.sort.bam \
                 -outdir results/qualimap_output/PacBio_clover_alignment_1_2_maphifi

qualimap bamqc -bam results/HIFI_alignment/PacBio_clover_alignment_1_2_sr.sort.bam \
                 -outdir results/qualimap_output/PacBio_clover_alignment_1_2_sr

Java memory size is set to 1200M
Launching application...

QualiMap v.2.3
Built on 2023-05-19 16:57

Selected tool: bamqc
Available memory (Mb): 35
Max memory (Mb): 1258
Starting bam qc....
Loading sam header...
Loading locator...
Loading reference...
Number of windows: 400, effective number of windows: 401
Chunk of reads size: 1000
Number of threads: 192
Processed 50 out of 401 windows...
Processed 100 out of 401 windows...
Processed 150 out of 401 windows...
Processed 200 out of 401 windows...
Processed 250 out of 401 windows...
Processed 300 out of 401 windows...
Processed 350 out of 401 windows...
Processed 400 out of 401 windows...
Total processed windows:401
Number of reads: 4395
Number of valid reads: 4810
Number of correct strand reads:0

Inside of regions...
Num mapped reads: 4395
Num mapped first of pair: 0
Num mapped second of pair: 0
Num singletons: 0
Time taken to analyze reads: 8
Computing descriptors...
numberOfMappedBases: 70053032
referenceSize: 2089554
numberOfSequenc

For easier comparison, we can again collapse the two reports into a single one using `MultiQC`, in the same way we did for putting together the other reports from `fastQC`.

In [21]:
%%bash

#run multiqc
multiqc --outdir results/qualimap_output results/qualimap_output


/// ]8;id=5364281;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /faststorage/project/ngssummer2024/2026/manuel/Intro-NGS-AU_course/Notebooks/results/qualimap_output
         searching | ━━━━━━━━━━━━━━━━━━━━m /PacBio_clover_alignment_1_2_maphi 0% 0/92  fi/css/jquery.js━━━━━━━━━━━━━━━━━━━━ 100% 92/92  
          qualimap | Found 2 BamQC reports
     write_results | Data        : results/qualimap_output/multiqc_data
     write_results | Report      : results/qualimap_output/multiqc_report.html
           multiqc | MultiQC complete


Now you can visualize the report generated, which is in `results/qualimap_output/multiqc_report.html`.

Next, we map the white clover PacBio Hifi reads to contig1 and contig2 separately, using the setting you selected at the previous step (let's say `map-hifi` was chosen, but you are free to change this setting in the commands). As the two contigs represent the two white clover subgenomes, this mapping will allow you to see the two subgenome haplotypes and call subgenome SNPs.


In [22]:
%%bash 
minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_1.sam \
                            reference_data/DNA_Contig1.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq

[M::mm_idx_gen::0.023*0.83] collected minimizers
[M::mm_idx_gen::0.064*1.75] sorted minimizers
[M::main::0.065*1.74] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::0.068*1.71] mid_occ = 50
[M::mm_idx_stat] kmer size: 19; skip: 19; is_hpc: 0; #seq: 1
[M::mm_idx_stat::0.069*1.69] distinct minimizers: 91166 (91.66% are singletons); average occurrences: 1.139; average spacing: 9.939; total length: 1031631
[M::worker_pipeline::27.434*2.95] mapped 4395 sequences
[M::main] Version: 2.31-r1302
[M::main] CMD: minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_1.sam reference_data/DNA_Contig1.fasta ../Data/Clover_Data/Hifi_reads_white_clover.fastq
[M::main] Real time: 27.442 sec; CPU: 80.852 sec; Peak RSS: 1.604 GB


In [23]:
%%bash 
minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_2.sam \
                            reference_data/DNA_Contig2.fasta \
                            ../Data/Clover_Data/Hifi_reads_white_clover.fastq

[M::mm_idx_gen::0.020*0.97] collected minimizers
[M::mm_idx_gen::0.054*1.79] sorted minimizers
[M::main::0.054*1.78] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::0.056*1.75] mid_occ = 50
[M::mm_idx_stat] kmer size: 19; skip: 19; is_hpc: 0; #seq: 1
[M::mm_idx_stat::0.057*1.74] distinct minimizers: 97892 (93.26% are singletons); average occurrences: 1.083; average spacing: 9.980; total length: 1057923
[M::worker_pipeline::27.566*2.95] mapped 4395 sequences
[M::main] Version: 2.31-r1302
[M::main] CMD: minimap2 -a -x map-hifi -o results/HIFI_alignment/PacBio_clover_alignment_2.sam reference_data/DNA_Contig2.fasta ../Data/Clover_Data/Hifi_reads_white_clover.fastq
[M::main] Real time: 27.580 sec; CPU: 81.303 sec; Peak RSS: 1.367 GB


Sort the bam files and create their index using `samtools`

In [24]:
%%bash
samtools sort results/HIFI_alignment/PacBio_clover_alignment_1.sam -o results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam
samtools sort results/HIFI_alignment/PacBio_clover_alignment_2.sam -o results/HIFI_alignment/PacBio_clover_alignment_2.sort.bam

In [25]:
%%bash
samtools index results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam
samtools index results/HIFI_alignment/PacBio_clover_alignment_2.sort.bam

Perform quality control

In [26]:
%%bash
mkdir -p results/qualimap_output
qualimap bamqc -bam results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam -outdir results/qualimap_output/PacBio_clover_alignment_1

Java memory size is set to 1200M
Launching application...

QualiMap v.2.3
Built on 2023-05-19 16:57

Selected tool: bamqc
Available memory (Mb): 35
Max memory (Mb): 1258
Starting bam qc....
Loading sam header...
Loading locator...
Loading reference...
Number of windows: 400, effective number of windows: 400
Chunk of reads size: 1000
Number of threads: 192
Processed 50 out of 400 windows...
Processed 100 out of 400 windows...
Processed 150 out of 400 windows...
Processed 200 out of 400 windows...
Processed 250 out of 400 windows...
Processed 300 out of 400 windows...
Processed 350 out of 400 windows...
Processed 400 out of 400 windows...
Total processed windows:400
Number of reads: 4395
Number of valid reads: 8792
Number of correct strand reads:0

Inside of regions...
Num mapped reads: 4337
Num mapped first of pair: 0
Num mapped second of pair: 0
Num singletons: 0
Time taken to analyze reads: 7
Computing descriptors...
numberOfMappedBases: 53994557
referenceSize: 1031631
numberOfSequenc

In [27]:
%%bash
qualimap bamqc -bam results/HIFI_alignment/PacBio_clover_alignment_2.sort.bam -outdir results/qualimap_output/PacBio_clover_alignment_2

Java memory size is set to 1200M
Launching application...

QualiMap v.2.3
Built on 2023-05-19 16:57

Selected tool: bamqc
Available memory (Mb): 35
Max memory (Mb): 1258
Starting bam qc....
Loading sam header...
Loading locator...
Loading reference...
Number of windows: 400, effective number of windows: 400
Chunk of reads size: 1000
Number of threads: 192
Processed 50 out of 400 windows...
Processed 100 out of 400 windows...
Processed 150 out of 400 windows...
Processed 200 out of 400 windows...
Processed 250 out of 400 windows...
Processed 300 out of 400 windows...
Processed 350 out of 400 windows...
Processed 400 out of 400 windows...
Total processed windows:400
Number of reads: 4395
Number of valid reads: 9163
Number of correct strand reads:0

Inside of regions...
Num mapped reads: 4393
Num mapped first of pair: 0
Num mapped second of pair: 0
Num singletons: 0
Time taken to analyze reads: 7
Computing descriptors...
numberOfMappedBases: 54944299
referenceSize: 1057923
numberOfSequenc

In [28]:
%%bash

#run multiqc
multiqc --outdir results/qualimap_output results/qualimap_output


/// ]8;id=12542088;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /faststorage/project/ngssummer2024/2026/manuel/Intro-NGS-AU_course/Notebooks/results/qualimap_output
         searching | ━━━━━━━━━━━━━━━━━━━━0m a_qualimapReport/mapped_reads_cli 0% 0/185  2m93/185 results/qualimap_output/multiqc_report.htmlpping_profile.t…━━━━━━━━━━━━━━━━━━━━ 100% 185/185  
          qualimap | Found 4 BamQC reports
     write_results | Existing reports found, adding suffix to filenames. Use '--force' to overwrite.
     write_results | Data        : results/qualimap_output/multiqc_data_1
     write_results | Report      : results/qualimap_output/multiqc_report_1.html
           multiqc | MultiQC complete


:::{.callout-tip title="Task: IGV visualization and Questions"}

Now you can inspect the alignment files in `IGV`. 

* First, you will need to download the reference fasta sequence in `../Data/Clover_Data/DNA_Contig1_2.fasta` and import it into IGV. You can do the same for the files `DNA_Contig1.fasta` and `DNA_Contig2.fasta` that you might need later. In IGV, this is done with the menu `Genomes --> Load Genome from file` menu and by selecting the relevant fasta file. Then, choose the reference you need from the drop-down menu (see figure below). 
![](./images/IGVref.png)
You will not yet see much, but you can choose one of the two subgenomes (contig 1 or 2) and double click on a chromosome position to inspect the reference sequence. The next step will visualize the mapped files on IGV.


* Each mapped genome can be seen in IGV against the reference file of choice. To load an aligned file, first download it together with the index file in `.bai` format. For example, you need to download both `results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam` and `results/HIFI_alignment/PacBio_clover_alignment_1.sort.bam.bai` to see this alignment (you need to open only the `.bam` file with IGV). If you open more files, their alignments will be distributed in the IGV interface, and you can change the size of each visualization yourself (below shown with only one opened alignment).
![](./images/IGVbam.png)

Now compare in IGV the two bam files `PacBio_clover_alignment_1.sort.bam` and `PacBio_clover_alignment_2.sort.bam`.

* What do you observe when comparing the two BAM files? 
* Have a look at the polymorphic regions in IGV. Are they true polymorphisms?

Add to the visualization the third alignment `PacBio_clover_alignment_1_2_maphifi.sort.bam` in IGV.

* Why do you see fluctuations in coverage and large regions without any apparent subgenome SNPs?
* What are the major differences between the stats for the reads mapped to Contigs1&2 versus contig1 and contig2? What is your interpretation of the differences?


:::

---

## RNA-seq mapping — short RNA reads

In the `../Data` folder you will find 24 RNA-seq libraries: 12 from the `S10` line and 12 from `Tienshan`. Each library is **paired-end**, which means that each library is split into two files—one for forward reads (R1) and one for reverse reads (R2). For example: `S10_1_1.R1.fastq` and `S10_1_1.R2.fastq`.

We align each library separately, then merge the 12 S10 alignments into one sample and the 12 Tienshan alignments into another. This gives us two final samples for comparison.


Before aligning reads, we need to prepare the reference genome for STAR. This involves two steps:

1. **Build a genome index** using `STAR --runMode genomeGenerate`. This creates a searchable index of the reference genome, allowing STAR to align reads quickly without scanning the whole sequence every time.

2. **Convert gene annotations** from `gff` to `gtf` format using `gffread`. STAR needs annotations in GTF format to count gene-level expression (how many reads map to each gene).

`STAR` is a very complex tool with many options, so it is always useful to [have a reference manual](https://physiology.med.cornell.edu/faculty/skrabanek/lab/angsd/lecture_notes/STARmanual.pdf).

In [31]:
%%bash
gffread -T -o reference_data/white_clover_genes.gtf ../Data/Clover_Data/white_clover_genes.gff

In [30]:
%%bash
STAR --runThreadN 8 \
--runMode genomeGenerate \
--genomeDir results/STAR_output/indexing_contigs_1_2 \
--genomeFastaFiles reference_data/DNA_Contig1_2.fasta \
--sjdbGTFfile reference_data/white_clover_genes.gtf 

	STAR --runThreadN 8 --runMode genomeGenerate --genomeDir results/STAR_output/indexing_contigs_1_2 --genomeFastaFiles reference_data/DNA_Contig1_2.fasta --sjdbGTFfile reference_data/white_clover_genes.gtf
	STAR version: 2.7.10a   compiled: 2022-01-14T18:50:00-05:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Jun 03 15:24:35 ..... started STAR run


!!!!! WARNING: Could not move Log.out file from ./Log.out into results/STAR_output/indexing_contigs_1_2/Log.out. Will keep ./Log.out



Jun 03 15:24:35 ... starting to generate Genome files
Jun 03 15:24:35 ..... processing annotations GTF


!!!!! WARNING: --genomeSAindexNbases 14 is too large for the genome size=2089554, which may cause seg-fault at the mapping step. Re-run genome generation with recommended --genomeSAindexNbases 9


Jun 03 15:24:35 ... starting to sort Suffix Array. This may take a long time...
Jun 03 15:24:35 ... sorting Suffix Array chunks and saving them to disk...
Jun 03 15:24:35 ... loading chunks from disk, packing SA...
Jun 03 15:24:35 ... finished generating suffix array
Jun 03 15:24:35 ... generating Suffix Array index
Jun 03 15:24:38 ... completed Suffix Array index
Jun 03 15:24:38 ..... inserting junctions into the genome indices
Jun 03 15:24:43 ... writing Genome to disk ...
Jun 03 15:24:43 ... writing Suffix Array to disk ...
Jun 03 15:24:43 ... writing SAindex to disk
Jun 03 15:24:44 ..... finished successfully


We got a warning saying
```
!!!!! WARNING: --genomeSAindexNbases 14 is too large for the genome size=2089554, which may cause seg-fault at the mapping step. Re-run genome generation with recommended --genomeSAindexNbases 9
```
meaning we need shorter strings of bases (9 bases instead of 14) to be indexed, as our reference genome is very short, and too long strings would cause many alignment errors. So we rerun the command with the suggested option (down below).

In [32]:
%%bash
STAR --runThreadN 8 \
--runMode genomeGenerate \
--genomeDir results/STAR_output/indexing_contigs_1_2 \
--genomeFastaFiles reference_data/DNA_Contig1_2.fasta \
--sjdbGTFfile reference_data/white_clover_genes.gtf \
--genomeSAindexNbases 9

	STAR --runThreadN 8 --runMode genomeGenerate --genomeDir results/STAR_output/indexing_contigs_1_2 --genomeFastaFiles reference_data/DNA_Contig1_2.fasta --sjdbGTFfile reference_data/white_clover_genes.gtf --genomeSAindexNbases 9
	STAR version: 2.7.10a   compiled: 2022-01-14T18:50:00-05:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Jun 03 15:39:19 ..... started STAR run


!!!!! WARNING: Could not move Log.out file from ./Log.out into results/STAR_output/indexing_contigs_1_2/Log.out. Will keep ./Log.out



Jun 03 15:39:19 ... starting to generate Genome files
Jun 03 15:39:19 ..... processing annotations GTF
Jun 03 15:39:19 ... starting to sort Suffix Array. This may take a long time...
Jun 03 15:39:19 ... sorting Suffix Array chunks and saving them to disk...
Jun 03 15:39:19 ... loading chunks from disk, packing SA...
Jun 03 15:39:20 ... finished generating suffix array
Jun 03 15:39:20 ... generating Suffix Array index
Jun 03 15:39:20 ... completed Suffix Array index
Jun 03 15:39:20 ..... inserting junctions into the genome indices
Jun 03 15:39:20 ... writing Genome to disk ...
Jun 03 15:39:20 ... writing Suffix Array to disk ...
Jun 03 15:39:20 ... writing SAindex to disk
Jun 03 15:39:20 ..... finished successfully


We use again `STAR` to align every single library for `S10`. We extract the library name of each file and run STAR through each pair of files. Note that plant introns are very rarely more than `5000 bp` and that you are mapping to two homoeologous contigs that show high similarity, especially in genic regions. We set the maximum size to 5000 using `--alignIntronMax 5000`.

In [33]:
%%bash
for i in `ls ../Data/Clover_Data/S10*.R1.fastq`
do

PREFIXNAME=`basename $i .R1.fastq`
echo "###############################################"
echo "##### ALIGNING PAIRED-END READS "$PREFIXNAME
echo "###############################################"
STAR --genomeDir results/STAR_output/indexing_contigs_1_2/ \
--runThreadN 8 \
--runMode alignReads \
--readFilesIn ../Data/Clover_Data/$PREFIXNAME.R1.fastq ../Data/Clover_Data/$PREFIXNAME.R2.fastq \
--outFileNamePrefix results/STAR_output/S10_align_contigs_1_2/$PREFIXNAME \
--outSAMtype BAM SortedByCoordinate \
--outSAMattributes Standard \
--quantMode GeneCounts \
--alignIntronMax 5000

done

###############################################
##### ALIGNING PAIRED-END READS S10_1_1
###############################################
	STAR --genomeDir results/STAR_output/indexing_contigs_1_2/ --runThreadN 8 --runMode alignReads --readFilesIn ../Data/Clover_Data/S10_1_1.R1.fastq ../Data/Clover_Data/S10_1_1.R2.fastq --outFileNamePrefix results/STAR_output/S10_align_contigs_1_2/S10_1_1 --outSAMtype BAM SortedByCoordinate --outSAMattributes Standard --quantMode GeneCounts --alignIntronMax 5000
	STAR version: 2.7.10a   compiled: 2022-01-14T18:50:00-05:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Jun 03 15:39:48 ..... started STAR run
Jun 03 15:39:48 ..... loading genome
Jun 03 15:39:48 ..... started mapping
Jun 03 15:39:53 ..... finished mapping
Jun 03 15:39:53 ..... started sorting BAM
Jun 03 15:39:54 ..... finished successfully
###############################################
##### ALIGNING PAIRED-END READS S10_1_2
###############################################
	STAR --genome

Do the same alignment for `Tienshan` libraries

In [34]:
%%bash
for i in `ls ../Data/Clover_Data/TI*.R1.fastq`
do

PREFIXNAME=`basename $i .R1.fastq`
echo "###############################################"
echo "##### ALIGNING PAIRED-END READS "$PREFIXNAME
echo "###############################################"
STAR --genomeDir results/STAR_output/indexing_contigs_1_2/ \
--runThreadN 8 \
--readFilesIn ../Data/Clover_Data/$PREFIXNAME.R1.fastq ../Data/Clover_Data/$PREFIXNAME.R2.fastq \
--outFileNamePrefix results/STAR_output/TI_align_contigs_1_2/$PREFIXNAME \
--outSAMtype BAM SortedByCoordinate \
--outSAMattributes Standard \
--quantMode GeneCounts \
--alignIntronMax 5000 

done

###############################################
##### ALIGNING PAIRED-END READS TI_1_1
###############################################
	STAR --genomeDir results/STAR_output/indexing_contigs_1_2/ --runThreadN 8 --readFilesIn ../Data/Clover_Data/TI_1_1.R1.fastq ../Data/Clover_Data/TI_1_1.R2.fastq --outFileNamePrefix results/STAR_output/TI_align_contigs_1_2/TI_1_1 --outSAMtype BAM SortedByCoordinate --outSAMattributes Standard --quantMode GeneCounts --alignIntronMax 5000
	STAR version: 2.7.10a   compiled: 2022-01-14T18:50:00-05:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Jun 03 15:40:30 ..... started STAR run
Jun 03 15:40:30 ..... loading genome
Jun 03 15:40:30 ..... started mapping
Jun 03 15:40:36 ..... finished mapping
Jun 03 15:40:36 ..... started sorting BAM
Jun 03 15:40:37 ..... finished successfully
###############################################
##### ALIGNING PAIRED-END READS TI_1_2
###############################################
	STAR --genomeDir results/STAR_output/ind

Run quality control on each aligned library with `MultiQC`. In this way there will be a whole report to compare `S10` files and `Tienshan` files.

In [35]:
%%bash
multiqc --outdir results/multiqc_output/TI_STAR_align_1_2 \
            results/STAR_output/TI_align_contigs_1_2/


/// ]8;id=4315966;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /faststorage/project/ngssummer2024/2026/manuel/Intro-NGS-AU_course/Notebooks/results/STAR_output/TI_align_contigs_1_2
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0/36  ━━━━━━━━━━━━━━━━━━━━ 100% 36/36  
              star | Found 6 reports and 6 gene count files
     write_results | Data        : results/multiqc_output/TI_STAR_align_1_2/multiqc_data
     write_results | Report      : results/multiqc_output/TI_STAR_align_1_2/multiqc_report.html
           multiqc | MultiQC complete


In [36]:
%%bash
multiqc --outdir results/multiqc_output/S10_STAR_align_1_2 \
            results/STAR_output/S10_align_contigs_1_2/


/// ]8;id=10465113;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /faststorage/project/ngssummer2024/2026/manuel/Intro-NGS-AU_course/Notebooks/results/STAR_output/S10_align_contigs_1_2
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0/36  ━━━━━━━━━━━━━━━━━━━━ 100% 36/36  
              star | Found 6 reports and 6 gene count files
     write_results | Data        : results/multiqc_output/S10_STAR_align_1_2/multiqc_data
     write_results | Report      : results/multiqc_output/S10_STAR_align_1_2/multiqc_report.html
           multiqc | MultiQC complete


We merge the outputs of each group of aligned libraries. Here is how the files look like for the `Tienshan`.

In [37]:
!ls -lh  results/STAR_output/TI_align_contigs_1_2/TI_*.sortedByCoord.out.bam 

-rw-rw-r-- 1 manuelpv ngssummer2024 6.6M Jun  3 15:40 results/STAR_output/TI_align_contigs_1_2/TI_1_1Aligned.sortedByCoord.out.bam
-rw-rw-r-- 1 manuelpv ngssummer2024 6.0M Jun  3 15:40 results/STAR_output/TI_align_contigs_1_2/TI_1_2Aligned.sortedByCoord.out.bam
-rw-rw-r-- 1 manuelpv ngssummer2024 7.1M Jun  3 15:40 results/STAR_output/TI_align_contigs_1_2/TI_1_3Aligned.sortedByCoord.out.bam
-rw-rw-r-- 1 manuelpv ngssummer2024 9.6M Jun  3 15:40 results/STAR_output/TI_align_contigs_1_2/TI_2_1Aligned.sortedByCoord.out.bam
-rw-rw-r-- 1 manuelpv ngssummer2024 7.2M Jun  3 15:41 results/STAR_output/TI_align_contigs_1_2/TI_2_2Aligned.sortedByCoord.out.bam
-rw-rw-r-- 1 manuelpv ngssummer2024 8.2M Jun  3 15:41 results/STAR_output/TI_align_contigs_1_2/TI_2_3Aligned.sortedByCoord.out.bam


Apply `samtools merge` to combine the individual per-library BAM files

In [38]:
%%bash
mkdir -p results/STAR_output/TI_align_contigs_1_2_merge/
samtools merge -f results/STAR_output/TI_align_contigs_1_2_merge/TI.sorted.bam results/STAR_output/TI_align_contigs_1_2/TI_*.sortedByCoord.out.bam 

In [39]:
%%bash
mkdir -p results/STAR_output/S10_align_contigs_1_2_merge/
samtools merge -f results/STAR_output/S10_align_contigs_1_2_merge/S10.sorted.bam results/STAR_output/S10_align_contigs_1_2/S10_*.sortedByCoord.out.bam 

Index both merging outputs. A file in format `bam.bai` will appear in their respective folders.

In [40]:
%%bash
samtools index results/STAR_output/S10_align_contigs_1_2_merge/S10.sorted.bam

In [41]:
%%bash
samtools index results/STAR_output/TI_align_contigs_1_2_merge/TI.sorted.bam

:::{.callout-note title="Wrapping up"}

In this exercise, you learned to align various types of data after performing quality control for raw data. We looked at some of the options for the aligners and at how to use some of the basic samtools manipulation programs. The outputs from the RNA alignments will be used for the VCF file analysis in the next notebook, and the RNA alignments will be use for the bulk RNA data analysis.

:::